In [1]:
# [셀1] import + 동결 상수
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

ID_COL, TARGET_COL, C20_COL = "C64", "C65", "C20"
FMT = "%Y-%m-%d %H:%M:%S.%f"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33", "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
SIGMA_C65 = 261.7

# 프로토콜 상수 (동결)
INIT_WEEKS = 4
H_DAYS = 7
MIN_TEST = 30            # 주간 fold 최소 test 웨이퍼
# 판정 문턱 (동결)
THR_RETRAIN_HELP = 3.0   # (i)
THR_VS_CORE = 0.3        # (ii)
THR_R2 = 0.9             # (iii)
THR_CATASTROPHE = 1.5    # (iv)

XGB_DEVICE = os.environ.get("XGB_DEVICE", "cpu")

def _find(name):
    for d in [".", "data", "colab_GA", "..", os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"), os.path.join("..", "modeling_v13", "colab_GA"),
              os.path.join("..", "문제1(하)")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz"); META = _find("core10_meta_wf.csv")
LEANJSON = _find("lean_timestable_set.json"); XGBJSON = _find("tuned_params_v13_xgb.json"); TRAIN = _find("train_data.csv")
print("xgboost", xgb.__version__, "| device", XGB_DEVICE, "| INIT_WEEKS", INIT_WEEKS, "| H", H_DAYS, "d")


xgboost 3.3.0 | device cpu | INIT_WEEKS 4 | H 7 d


In [2]:
# [셀2] 헬퍼
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

def make_xgb(params, rounds):
    p = dict(params)
    p.update(objective="reg:squarederror", tree_method="hist", device=XGB_DEVICE,
             random_state=42, n_estimators=int(rounds))
    return xgb.XGBRegressor(**p)


In [3]:
# [셀3] 로드 + lean-85/core10 + 웨이퍼·Lot 시간 + asserts
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
tt = pd.read_csv(TRAIN, usecols=[ID_COL, "C40"]); tt[ID_COL] = tt[ID_COL].astype(str)
tt["_ts"] = pd.to_datetime(tt["C40"], format=FMT)
df = df.merge(tt.groupby(ID_COL)["_ts"].min().reset_index().rename(columns={"_ts": "wf_ts"}), on=ID_COL, how="inner").reset_index(drop=True)
y = df[TARGET_COL].to_numpy(float)

LEAN = json.load(open(LEANJSON, encoding="utf-8"))["lean_features"]
CORE = [f for f in CORE10 if f in df.columns]
xgj = json.load(open(XGBJSON, encoding="utf-8")); XGB_PARAMS = xgj["params"]; XGB_ROUNDS = int(xgj["n_estimators"])

ok, have = floor_ok(LEAN)
assert all(f in df.columns for f in LEAN), "lean 누락 피처"
assert C20_COL not in LEAN and ID_COL not in LEAN and "fold_kf5" not in LEAN, "누수 피처!"
assert ok, f"lean floor 위반 {have}"

# Lot 대표 시각 = Lot 내 웨이퍼 median → Lot 단위 시간정렬
lot_time = df.groupby(C20_COL)["wf_ts"].median()
df["lot_ts"] = df[C20_COL].map(lot_time)
print(f"lean n={len(LEAN)} floor={have} | core10 n={len(CORE)} | df {df.shape}")
print(f"기간 {str(df['wf_ts'].min())[:10]} ~ {str(df['wf_ts'].max())[:10]} | XGB {XGB_ROUNDS}r")


lean n=85 floor={'C17': 2, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 1} | core10 n=10 | df (11939, 669)
기간 2018-12-01 ~ 2019-02-08 | XGB 246r


In [4]:
# [셀4] 확장창 워크포워드 (3 arm, 인과적)
Xl = df[LEAN]; Xc = df[CORE]
lot_ts = df["lot_ts"]
t0 = df["wf_ts"].min(); init_end = t0 + pd.Timedelta(weeks=INIT_WEEKS); end = df["wf_ts"].max()
H = pd.Timedelta(days=H_DAYS)

init_idx = np.where((lot_ts <= init_end).to_numpy())[0]
m_nr = make_xgb(XGB_PARAMS, XGB_ROUNDS); m_nr.fit(Xl.iloc[init_idx], y[init_idx])   # 무재학습 = 1회 학습
print(f"초기 학습창 n={len(init_idx)} (~{str(init_end)[:10]}) | 무재학습 모델 고정")

folds = []; PY = {"lean": [], "nr": [], "core": []}; PP = {"lean": [], "nr": [], "core": []}
T = init_end
while T < end:
    te = ((lot_ts > T) & (lot_ts <= T + H)).to_numpy()
    te_idx = np.where(te)[0]
    if len(te_idx) < MIN_TEST:
        T = T + H; continue
    tr_idx = np.where((lot_ts <= T).to_numpy())[0]           # 확장창
    yte = y[te_idx]
    ma = make_xgb(XGB_PARAMS, XGB_ROUNDS); ma.fit(Xl.iloc[tr_idx], y[tr_idx]); pa = ma.predict(Xl.iloc[te_idx])  # (a) 재학습 lean
    mc = make_xgb(XGB_PARAMS, XGB_ROUNDS); mc.fit(Xc.iloc[tr_idx], y[tr_idx]); pc = mc.predict(Xc.iloc[te_idx])  # (c) 재학습 core
    pb = m_nr.predict(Xl.iloc[te_idx])                                                                            # (b) 무재학습
    folds.append(dict(cut=str(T)[:10], n=int(len(te_idx)), train_n=int(len(tr_idx)),
                      retrain_lean=round(_rmse(yte, pa), 3), noretrain=round(_rmse(yte, pb), 3), retrain_core=round(_rmse(yte, pc), 3)))
    for k, p in [("lean", pa), ("nr", pb), ("core", pc)]:
        PY[k].append(yte); PP[k].append(p)
    print(f"  {str(T)[:10]} (test {len(te_idx):>4}, train {len(tr_idx):>5}): 재학습 {_rmse(yte,pa):6.1f} | 무재학습 {_rmse(yte,pb):6.1f} | core10재 {_rmse(yte,pc):6.1f}")
    T = T + H
print(f"완료: {len(folds)} 주간 fold")


초기 학습창 n=4661 (~2018-12-29) | 무재학습 모델 고정
  2018-12-29 (test  834, train  4661): 재학습  147.0 | 무재학습  147.0 | core10재  141.8
  2019-01-05 (test 1077, train  5495): 재학습  113.3 | 무재학습  168.3 | core10재  106.2
  2019-01-12 (test 1211, train  6572): 재학습   75.2 | 무재학습  216.0 | core10재   73.0
  2019-01-19 (test 1475, train  7783): 재학습  109.5 | 무재학습  282.7 | core10재  113.5
  2019-01-26 (test 1408, train  9258): 재학습   68.1 | 무재학습  292.4 | core10재   68.9
  2019-02-02 (test 1273, train 10666): 재학습   87.0 | 무재학습  319.1 | core10재   85.7
완료: 6 주간 fold


In [5]:
# [셀5] pooled 집계 + 판정 G6′-B2′ (참고 — 공식은 강건)
def pooled(k):
    return _rmse(np.concatenate(PY[k]), np.concatenate(PP[k]))
p_lean, p_nr, p_core = pooled("lean"), pooled("nr"), pooled("core")
r2 = r2_honest(p_lean)
worst_fold = max(f["retrain_lean"] for f in folds)

print("=== pooled (전 fold 통합) ===")
print(f"  재학습 lean-85 = {p_lean:.3f} (R²={r2}) | 무재학습 = {p_nr:.3f} | 재학습 core10 = {p_core:.3f}")

c_i = p_lean < p_nr - THR_RETRAIN_HELP
c_ii = p_lean <= p_core + THR_VS_CORE
c_iii = r2 >= THR_R2
c_iv = worst_fold <= THR_CATASTROPHE * p_lean
print("\n=== 판정 G6′-B2′ (참고) ===")
print(f"  (i)   재학습 < 무재학습 −{THR_RETRAIN_HELP:g}:  {p_lean:.2f} < {p_nr-THR_RETRAIN_HELP:.2f}  → {'PASS' if c_i else 'FAIL'}")
print(f"  (ii)  재학습 lean ≤ core10 +{THR_VS_CORE:g}:  {p_lean:.2f} ≤ {p_core+THR_VS_CORE:.2f}  → {'PASS' if c_ii else 'FAIL'}")
print(f"  (iii) pooled R² ≥ {THR_R2}:  {r2}  → {'PASS' if c_iii else 'FAIL'}")
print(f"  (iv)  파국 fold 없음:  최악 {worst_fold:.1f} ≤ {THR_CATASTROPHE*p_lean:.1f}  → {'PASS' if c_iv else 'FAIL'}")
print(f"  → B2′ 잠정: {'PASS 권고 → M8' if all([c_i,c_ii,c_iii,c_iv]) else 'MIXED/FAIL — 강건 판정'}")


=== pooled (전 fold 통합) ===
  재학습 lean-85 = 99.840 (R²=0.8545) | 무재학습 = 254.899 | 재학습 core10 = 98.361

=== 판정 G6′-B2′ (참고) ===
  (i)   재학습 < 무재학습 −3:  99.84 < 251.90  → PASS
  (ii)  재학습 lean ≤ core10 +0.3:  99.84 ≤ 98.66  → FAIL
  (iii) pooled R² ≥ 0.9:  0.8545  → FAIL
  (iv)  파국 fold 없음:  최악 147.0 ≤ 149.8  → PASS
  → B2′ 잠정: MIXED/FAIL — 강건 판정


In [6]:
# [셀6] 저장
out = dict(protocol=f"expanding walk-forward init={INIT_WEEKS}wk H={H_DAYS}d",
           pooled=dict(retrain_lean=round(p_lean, 3), noretrain=round(p_nr, 3), retrain_core=round(p_core, 3), R2=r2),
           gates=dict(retrain_helps=bool(c_i), vs_core=bool(c_ii), r2=bool(c_iii), no_catastrophe=bool(c_iv),
                      all_pass=bool(c_i and c_ii and c_iii and c_iv)),
           worst_fold=worst_fold, n_folds=len(folds), folds=folds,
           thresholds=dict(retrain_help=THR_RETRAIN_HELP, vs_core=THR_VS_CORE, r2=THR_R2, catastrophe=THR_CATASTROPHE),
           holdout="R5 2/2회차 소진", note="공식 G6′-B2′ 판정은 강건이 회신 숫자로.")
json.dump(out, open("b2prime_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved: b2prime_results.json")


saved: b2prime_results.json
